# HDAT DS 독립 종합문제 — 차량 센서 시계열 (Starter)

> 현대자동차그룹 또는 현대엔지비의 공식·복원 문제가 아닌 **독립 창작 연습문제**입니다. 외부 데이터 없이 합성 데이터를 만들고, 모델링은 PyTorch만 사용합니다.

과거 24시점의 차량 센서로 다음 시점의 위험 여부를 예측합니다. **시간 분할 → train-only 표준화 → window → baseline → 작은 GRU → validation → 저장·재로딩·submission**을 한 번에 연습합니다.

## 환경과 운영 습관

- 권장: Python 3.10–3.12, PyTorch 2.2 이상, JupyterLab/Notebook 7 이상
- CPU 전용, 약 900시점과 6 epoch라 일반 노트북에서 빠르게 끝납니다.
- 시작 전 **Kernel → Restart Kernel and Clear Outputs**, 각 큰 단계 후 **Ctrl/Cmd+S**를 누르세요.
- 제출 전에는 Restart + Run All로 실행 순서 의존성을 제거하고, CSV를 다시 읽어 행 수·열 이름·NaN을 확인하세요.
- Starter의 TODO를 채우기 전 `assert`가 실패하는 것은 정상입니다. 해설판은 한 가지 참고 풀이일 뿐입니다.

In [ ]:
import csv
import platform
import random
import tempfile
from pathlib import Path

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

print('Python :', platform.python_version(), '(권장 3.10–3.12)')
print('PyTorch:', torch.__version__, '(권장 2.2+)')
SEED = 2026
random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)
device = torch.device('cpu')

## 데이터 명세

특징 순서는 `speed`, `rpm`, `coolant`, `vibration`, `throttle`입니다. 레이블 1은 해당 시점의 합성 위험 신호가 train 구간 기준 상위 수준이라는 뜻입니다. 실제 차량 진단 기준이 아니며 학습용으로만 사용합니다.

In [ ]:
def make_vehicle_sensor_series(n_steps=900, seed=SEED):
    g = torch.Generator().manual_seed(seed)
    t = torch.arange(n_steps, dtype=torch.float32)

    def noise(scale):
        return scale * torch.randn(n_steps, generator=g)

    throttle = (0.45 + 0.24 * torch.sin(t / 19) + noise(0.06)).clamp(0, 1)
    speed = (54 + 17 * torch.sin(t / 31) + 8 * throttle + noise(2.5)).clamp_min(0)
    rpm = 650 + 27 * speed + 720 * throttle + noise(90)
    coolant = 79 + 0.012 * t + 4 * torch.sin(t / 83) + 2.5 * throttle + noise(0.8)
    vibration = 0.18 + 0.00016 * rpm + 0.09 * torch.relu(torch.sin(t / 11)) + noise(0.025)
    features = torch.stack([speed, rpm, coolant, vibration, throttle], dim=1).to(torch.float32)

    risk = (
        0.10 * (coolant - 82)
        + 2.4 * (vibration - 0.45)
        + 0.00045 * (rpm - 2100)
        + 0.7 * throttle
        + noise(0.35)
    )
    train_boundary = int(0.60 * n_steps)
    threshold = torch.quantile(risk[:train_boundary], 0.72)
    labels = (risk > threshold).to(torch.float32)
    return features, labels

feature_names = ['speed', 'rpm', 'coolant', 'vibration', 'throttle']
raw_X, raw_y = make_vehicle_sensor_series()
assert raw_X.shape == (900, 5) and raw_y.shape == (900,)
assert raw_X.dtype == torch.float32 and raw_y.dtype == torch.float32
assert torch.isfinite(raw_X).all() and torch.isfinite(raw_y).all()
print('raw:', raw_X.shape, raw_y.shape, 'positive rate=', round(raw_y.mean().item(), 3))

## 1. 시간 순서 분할

앞 60%를 train, 다음 20%를 validation, 마지막 20%를 test로 나누세요. 무작위 permutation은 금지합니다.

In [ ]:
# TODO 1: 540/180/180 시점으로 순서 분할하세요.
X_train_raw = y_train_raw = None
X_val_raw = y_val_raw = None
X_test_raw = y_test_raw = None

assert X_train_raw.shape == (540, 5) and y_train_raw.shape == (540,)
assert X_val_raw.shape == (180, 5) and y_val_raw.shape == (180,)
assert X_test_raw.shape == (180, 5) and y_test_raw.shape == (180,)
assert torch.equal(X_val_raw[0], raw_X[540])

## 2. train-only 표준화

특징별 평균·표준편차는 train에서만 구합니다. validation/test에도 그 값을 재사용하세요. 레이블은 표준화하지 않습니다.

In [ ]:
# TODO 2: dim=0, keepdim=True, unbiased=False를 사용하세요.
train_mean = train_std = None
X_train_n = X_val_n = X_test_n = None

assert train_mean.shape == (1, 5) and train_std.shape == (1, 5)
assert X_train_n.shape == (540, 5) and X_val_n.shape == (180, 5)
assert torch.all(train_std > 0)
assert torch.allclose(X_train_n.mean(0), torch.zeros(5), atol=1e-5)

## 3. 시계열 window 만들기

길이 24의 `[window, feature]` 입력으로 바로 다음 시점(`horizon=1`)의 레이블을 예측합니다. 각 split 안에서 따로 window를 만들어 경계를 넘지 않게 하세요.

In [ ]:
WINDOW = 24
HORIZON = 1

def make_windows(features, labels, window=WINDOW, horizon=HORIZON):
    # TODO 3: X는 [N_window, window, F], y는 [N_window, 1]을 반환하세요.
    raise NotImplementedError

X_train, y_train = make_windows(X_train_n, y_train_raw)
X_val, y_val = make_windows(X_val_n, y_val_raw)
X_test, y_test = make_windows(X_test_n, y_test_raw)

assert X_train.shape == (516, 24, 5) and y_train.shape == (516, 1)
assert X_val.shape == (156, 24, 5) and y_val.shape == (156, 1)
assert X_test.shape == (156, 24, 5) and y_test.shape == (156, 1)
assert X_train.dtype == torch.float32 and y_train.dtype == torch.float32

## 4. Majority baseline

train 레이블에서 다수 class 하나만 정하고 validation 전체에 예측하세요. 복잡한 모델은 최소한 이 기준과 비교해야 합니다.

In [ ]:
# TODO 4: train positive rate로 majority class와 val 정확도를 구하세요.
majority_class = None
baseline_val_accuracy = None

assert majority_class in (0.0, 1.0)
assert isinstance(baseline_val_accuracy, float)
assert 0.0 <= baseline_val_accuracy <= 1.0
print(f'baseline val accuracy: {baseline_val_accuracy:.3f}')

## 5. 작은 PyTorch GRU

입력 `[B, T, F]`를 hidden size 16의 GRU에 넣고 마지막 hidden state로 logit `[B, 1]`을 출력하세요. sigmoid는 모델에 넣지 않습니다.

In [ ]:
# TODO 5: VehicleRiskGRU를 완성하세요.
class VehicleRiskGRU(nn.Module):
    def __init__(self, n_features=5, hidden_size=16):
        super().__init__()
        raise NotImplementedError('TODO: GRU와 Linear를 정의하세요')

    def forward(self, x):
        raise NotImplementedError('TODO: 마지막 hidden state를 분류하세요')

model = VehicleRiskGRU().to(device)
sample_logits = model(torch.zeros(4, WINDOW, 5))
assert sample_logits.shape == (4, 1) and sample_logits.dtype == torch.float32

## 6. DataLoader와 학습

batch size 64, Adam, `BCEWithLogitsLoss`로 6 epoch 학습하세요. train만 shuffle하고 매 epoch 평균 loss를 기록합니다.

In [ ]:
# TODO 6: DataLoader, criterion, optimizer, 6-epoch loop를 작성하세요.
train_loader = val_loader = None
criterion = optimizer = None
train_losses = []

assert len(train_losses) == 6
assert all(isinstance(v, float) and torch.isfinite(torch.tensor(v)) for v in train_losses)
print('train losses:', [round(v, 4) for v in train_losses])

## 7. Validation metrics

`eval`과 `inference_mode`로 accuracy, precision, recall, F1을 직접 계산하세요. 분모가 0이면 작은 epsilon으로 보호합니다. 기준 threshold는 확률 0.5, 즉 logit 0입니다.

In [ ]:
# TODO 7: evaluate_binary(model, X, y)를 완성하세요.
def evaluate_binary(model, X, y):
    raise NotImplementedError

val_metrics = evaluate_binary(model, X_val, y_val)
assert set(val_metrics) == {'accuracy', 'precision', 'recall', 'f1'}
assert all(isinstance(v, float) and 0.0 <= v <= 1.0 for v in val_metrics.values())
print('baseline:', round(baseline_val_accuracy, 3), 'model:', val_metrics)

## 8. 체크포인트 재로딩과 submission 검증

`state_dict`, 정규화 통계, window 설정을 저장하세요. 새 모델에 재로딩한 뒤 test 확률을 `sample_id,risk_probability` CSV로 쓰고 다시 읽어 계약을 확인합니다.

In [ ]:
# TODO 8: 저장 → 새 객체 재로딩 → 확률 예측 → CSV 쓰기/읽기를 구현하세요.
tmp_dir = tempfile.TemporaryDirectory()
checkpoint_path = Path(tmp_dir.name) / 'vehicle_gru.pt'
submission_path = Path(tmp_dir.name) / 'submission.csv'
reloaded_model = None
test_probability = None
reloaded_rows = []

assert checkpoint_path.exists() and submission_path.exists()
assert isinstance(reloaded_model, VehicleRiskGRU)
assert test_probability.shape == (156, 1)
assert torch.isfinite(test_probability).all()
assert torch.all((0 <= test_probability) & (test_probability <= 1))
assert len(reloaded_rows) == 156
assert set(reloaded_rows[0]) == {'sample_id', 'risk_probability'}
print('submission contract OK:', submission_path)

## 제출 전 체크리스트

- [ ] 시간 순서를 지켰고 window가 split 경계를 넘지 않는다.
- [ ] 평균·표준편차는 train만으로 계산했다.
- [ ] baseline과 모델 validation 지표를 함께 기록했다.
- [ ] `eval` + inference/no-grad 문맥으로 예측했다.
- [ ] 저장한 새 모델로 submission을 다시 만들 수 있다.
- [ ] CSV를 재로딩해 열 이름, 행 수, 확률 범위를 확인했다.
- [ ] Kernel Restart + Run All 후 Ctrl/Cmd+S 했다.